### 🎯 Module Overview
This module covers everything you need to know about parsing and ingesting data for RAG systems, from basic text files to complex PDFs and databases. We'll use LangChain v0.3 and explore each technique with practical examples.

Table of Contents

- Introduction to Data Ingestion
- Text Files (.txt)
- PDF Documents
- Microsoft Word Documents
- CSV and Excel Files
- JSON and Structured Data
- Web Scraping
- Databases (SQL)
- Audio and Video Transcripts
- Advanced Techniques
- Best Practices

### 1) Introduction to Data Ingestion
for every document we read, we need to convert that into a document structure i.e. page_content and metadata where page_content is the main content while the metadata contains the information regarding that text. 


In [7]:
import os
from typing import List, Dict, Any
import pandas as pd

In [8]:
import langchain

In [13]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter, 
    CharacterTextSplitter,
    TokenTextSplitter)
print("Setup Completed!")

Setup Completed!


### Understanding Document Structure in Langchain

In [ ]:
## create a simple document
doc=Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source": "example.txt", 
        "page":1,
        "author":"Hemant Dhingra",
        "date_created":"2024-10-07",
        "custom_field":"any_value",
    }
)
print("Document Structure")
print(f"Page Content :{doc.page_content}")
print(f"Metadata :{doc.metadata}")

Document Structure
Page Content :This is the main text content that will be embedded and searched.
Metadata :{'source': 'example.txt', 'page': 1, 'author': 'Hemant Dhingra', 'date_created': '2024-10-07', 'custom_field': 'any_value'}


In [ ]:
# Why metadata matters:
print("\n📝 Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")


📝 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


When we use various document loaders of langchain, reading the files and loading will be done in the form of such document classes

In [ ]:
type(doc)

langchain_core.documents.base.Document

### 2. Text Files (.txt)

In [ ]:
### create 2 simple txt files
import os
os.makedirs("data/text_files", exist_ok=True)

In [ ]:
sample_texts={
    "data/text_files/RAG_intro.txt":"""Retrieval-Augmented Generation (RAG) is a powerful AI framework that enhances the capabilities of large language models (LLMs) by combining the best of two worlds — retrieval and generation. Traditional LLMs rely solely on the data they were trained on, meaning their knowledge is static and limited to a specific cutoff date. RAG solves this limitation by allowing models to access external information sources (like databases, documents, or the web) in real time.

Here’s how it works: when you ask a question, the system first retrieves the most relevant information from a connected knowledge base (this is the retrieval step). Then, instead of simply repeating that information, the LLM generates a coherent and contextually accurate response using both its internal understanding and the retrieved data.

This approach makes RAG particularly useful for tasks that require up-to-date, factual, or domain-specific knowledge, such as answering technical questions, summarizing research papers, or assisting customer support systems. By grounding its answers in real, retrievable sources, RAG also reduces hallucinations — instances where models produce false or fabricated details.

In essence, RAG transforms a language model from a “closed-book” learner into an “open-book” one — capable of reasoning with external information to deliver more accurate, transparent, and trustworthy results. As AI continues to evolve, RAG stands out as a key step toward making generative systems more intelligent and reliable in real-world applications. """, 

"data/text_files/Agents_intro.txt":"""LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text — they can reason, plan, and take actions to achieve specific goals. Unlike traditional chatbots that only respond to prompts, LLM Agents can interact with external tools, APIs, or databases, making them capable of performing real tasks such as searching the web, analyzing data, or automating workflows.

They operate using a “think–act–learn” loop: the model interprets a goal, decides what actions to take, executes them, and updates its understanding based on the results. This enables dynamic problem-solving, not just static conversation.

In short, LLM Agents represent the next evolution of AI — from passive responders to active collaborators — capable of working autonomously or alongside humans to perform complex, multi-step tasks efficiently and intelligently across industries like research, business automation, and customer support.



"""

}
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
print("Sample text files created!")

Sample text files created!


### Text loader - Read single File 

In [ ]:
from langchain_community.document_loaders import TextLoader

# Loading/reading a single text file
loader=TextLoader("data/text_files/RAG_intro.txt", encoding='utf-8')
loader

In [ ]:
documents=loader.load()
print(documents)


## This clearly means whenever you use the standard loaders of langchain_community you will always read the files as a page_content, metadata pair

[Document(metadata={'source': 'data/text_files/RAG_intro.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful AI framework that enhances the capabilities of large language models (LLMs) by combining the best of two worlds — retrieval and generation. Traditional LLMs rely solely on the data they were trained on, meaning their knowledge is static and limited to a specific cutoff date. RAG solves this limitation by allowing models to access external information sources (like databases, documents, or the web) in real time.\n\nHere’s how it works: when you ask a question, the system first retrieves the most relevant information from a connected knowledge base (this is the retrieval step). Then, instead of simply repeating that information, the LLM generates a coherent and contextually accurate response using both its internal understanding and the retrieved data.\n\nThis approach makes RAG particularly useful for tasks that require up-to-date, factual, or domain-specific 

In [ ]:
print(f"Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

Loaded 1 document
Content preview: Retrieval-Augmented Generation (RAG) is a powerful AI framework that enhances the capabilities of la...
Metadata: {'source': 'data/text_files/RAG_intro.txt'}


### Directory loader : Used when you have multiple  files

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
## Load all text files present inside a particular directory
dir_loader=DirectoryLoader(
    "data/text_files", 
    glob="**/*.txt", ## Pattern to match files : folder/anything.txt
    loader_cls = TextLoader, ## Loader class to use
    loader_kwargs = {'encoding' : 'utf-8'},
    show_progress =True
    )

documents = dir_loader.load()

100%|██████████| 2/2 [00:00<00:00, 124.69it/s]


In [ ]:
print(f"Loaded {len(documents)} documents")

for i, doc in enumerate(documents):
    print(f"Document : {i+1}")
    print(f"Content preview: {documents[i].page_content[:100]}...")
    print(f"Metadata: {documents[i].metadata}")
    print(f"Word Count: {len(documents[i].page_content)}")
    print("\n")

Loaded 2 documents
Document : 1
Content preview: LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text...
Metadata: {'source': 'data\\text_files\\Agents_intro.txt'}
Word Count: 953


Document : 2
Content preview: Retrieval-Augmented Generation (RAG) is a powerful AI framework that enhances the capabilities of la...
Metadata: {'source': 'data\\text_files\\RAG_intro.txt'}
Word Count: 1543




In [ ]:
#  Analysis
print("\n DirectoryLoader Characteristics:")
print(" Advantages:")
print("  - Loads multiple files at once")
print("  - Supports glob patterns")
print("  - Progress tracking")
print("  - Recursive directory scanning")

print("\n Disadvantages:")
print("  - All files must be same type")
print("  - Limited error handling per file")
print("  - Can be memory intensive for large directories")


 DirectoryLoader Characteristics:
 Advantages:
  - Loads multiple files at once
  - Supports glob patterns
  - Progress tracking
  - Recursive directory scanning

 Disadvantages:
  - All files must be same type
  - Limited error handling per file
  - Can be memory intensive for large directories


When you read any kind of file like Webpage, PDFs, Databases, etc. we get the return type as a Document and then we have a document Splitter also called as text splitter whose main aim is to split the document/text into chunks

### Text Splitting Strategies

In [ ]:
# different text splitters loaded already above


documents

[Document(metadata={'source': 'data\\text_files\\Agents_intro.txt'}, page_content='LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text — they can reason, plan, and take actions to achieve specific goals. Unlike traditional chatbots that only respond to prompts, LLM Agents can interact with external tools, APIs, or databases, making them capable of performing real tasks such as searching the web, analyzing data, or automating workflows.\n\nThey operate using a “think–act–learn” loop: the model interprets a goal, decides what actions to take, executes them, and updates its understanding based on the results. This enables dynamic problem-solving, not just static conversation.\n\nIn short, LLM Agents represent the next evolution of AI — from passive responders to active collaborators — capable of working autonomously or alongside humans to perform complex, multi-step tasks efficiently and intelligently across industries like research, busine

In [ ]:
text1=documents[0].page_content
text1

'LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text — they can reason, plan, and take actions to achieve specific goals. Unlike traditional chatbots that only respond to prompts, LLM Agents can interact with external tools, APIs, or databases, making them capable of performing real tasks such as searching the web, analyzing data, or automating workflows.\n\nThey operate using a “think–act–learn” loop: the model interprets a goal, decides what actions to take, executes them, and updates its understanding based on the results. This enables dynamic problem-solving, not just static conversation.\n\nIn short, LLM Agents represent the next evolution of AI — from passive responders to active collaborators — capable of working autonomously or alongside humans to perform complex, multi-step tasks efficiently and intelligently across industries like research, business automation, and customer support.\n\n\n\n'

In [ ]:
# 1) Character Text splitter

print("CHARACTER TEXT SPLITTER")
char_splitter=CharacterTextSplitter(
    separator=" ", # Split on new lines
    chunk_size=200, # Max chunk size in characters
    chunk_overlap=20, # Overlap between the chunks
    length_function=len # how to measure the chunk size
    )



char_chunks=char_splitter.split_text(text1)

print(text1)

print(f'created {len(char_chunks)} chunks')

print(f'first chunk : {char_chunks[0]}....')
print(f'second chunk : {char_chunks[1]}....')
print(f'third chunk : {char_chunks[2]}....')

## observe the obverlap after the splitting is done on the basis of spaces


CHARACTER TEXT SPLITTER
LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text — they can reason, plan, and take actions to achieve specific goals. Unlike traditional chatbots that only respond to prompts, LLM Agents can interact with external tools, APIs, or databases, making them capable of performing real tasks such as searching the web, analyzing data, or automating workflows.

They operate using a “think–act–learn” loop: the model interprets a goal, decides what actions to take, executes them, and updates its understanding based on the results. This enables dynamic problem-solving, not just static conversation.

In short, LLM Agents represent the next evolution of AI — from passive responders to active collaborators — capable of working autonomously or alongside humans to perform complex, multi-step tasks efficiently and intelligently across industries like research, business automation, and customer support.




created 6 chunks
first

In [ ]:
# 1) Character Text splitter

print("CHARACTER TEXT SPLITTER")
char_splitter=CharacterTextSplitter(
    separator="\n", # Split on new lines
    chunk_size=200, # Max chunk size in characters
    chunk_overlap=20, # Overlap between the chunks
    length_function=len # how to measure the chunk size
    )

char_chunks=char_splitter.split_text(text1)
print(text1)
print(f'created {len(char_chunks)} chunks')

print(f'first chunk : {char_chunks[0]}....')
print(f'second chunk : {char_chunks[1]}....')
print(f'third chunk : {char_chunks[2]}....')

## overlap wasnt observed here because the chunking basis 200 characters wasn't done

## chunks should ideally have been of the specified limit of 200 but since the \n wasn't found untill 200 characters, chunks were not made properly


Created a chunk of size 405, which is longer than the specified 200
Created a chunk of size 238, which is longer than the specified 200


CHARACTER TEXT SPLITTER
LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text — they can reason, plan, and take actions to achieve specific goals. Unlike traditional chatbots that only respond to prompts, LLM Agents can interact with external tools, APIs, or databases, making them capable of performing real tasks such as searching the web, analyzing data, or automating workflows.

They operate using a “think–act–learn” loop: the model interprets a goal, decides what actions to take, executes them, and updates its understanding based on the results. This enables dynamic problem-solving, not just static conversation.

In short, LLM Agents represent the next evolution of AI — from passive responders to active collaborators — capable of working autonomously or alongside humans to perform complex, multi-step tasks efficiently and intelligently across industries like research, business automation, and customer support.




created 3 chunks
first

In [ ]:
# 2) Recursive Text splitter : MOST RECOMMENDED

print("RECURSIVE TEXT SPLITTER")
recursive_splitter=RecursiveCharacterTextSplitter(
    separators=[" "], # Split basis these separators in order
    chunk_size=200, # Max chunk size in characters
    chunk_overlap=20, # Overlap between the chunks
    length_function=len # how to measure the chunk size
    )

recursive_chunks=recursive_splitter.split_text(text1)

print(text1)
print(f'created {len(recursive_chunks)} chunks')

print(f'first chunk : {recursive_chunks[0]}....')
print(f'second chunk : {recursive_chunks[1]}....')
print(f'third chunk : {recursive_chunks[2]}....')

## observe the overlap when the chunking was done basis the spaces

RECURSIVE TEXT SPLITTER
LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text — they can reason, plan, and take actions to achieve specific goals. Unlike traditional chatbots that only respond to prompts, LLM Agents can interact with external tools, APIs, or databases, making them capable of performing real tasks such as searching the web, analyzing data, or automating workflows.

They operate using a “think–act–learn” loop: the model interprets a goal, decides what actions to take, executes them, and updates its understanding based on the results. This enables dynamic problem-solving, not just static conversation.

In short, LLM Agents represent the next evolution of AI — from passive responders to active collaborators — capable of working autonomously or alongside humans to perform complex, multi-step tasks efficiently and intelligently across industries like research, business automation, and customer support.




created 6 chunks
first

In [ ]:
# 2) Recursive Text splitter : MOST RECOMMENDED

print("RECURSIVE TEXT SPLITTER")
recursive_splitter=RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ''], # Split basis these separators in order
    chunk_size=200, # Max chunk size in characters
    chunk_overlap=20, # Overlap between the chunks
    length_function=len # how to measure the chunk size
    )


recursive_chunks=recursive_splitter.split_text(text1)
print(text1)
print(f'created {len(recursive_chunks)} chunks')

print(f'first chunk : {recursive_chunks[0]}....')
print(f'second chunk : {recursive_chunks[1]}....')
print(f'third chunk : {recursive_chunks[2]}....')



RECURSIVE TEXT SPLITTER
LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text — they can reason, plan, and take actions to achieve specific goals. Unlike traditional chatbots that only respond to prompts, LLM Agents can interact with external tools, APIs, or databases, making them capable of performing real tasks such as searching the web, analyzing data, or automating workflows.

They operate using a “think–act–learn” loop: the model interprets a goal, decides what actions to take, executes them, and updates its understanding based on the results. This enables dynamic problem-solving, not just static conversation.

In short, LLM Agents represent the next evolution of AI — from passive responders to active collaborators — capable of working autonomously or alongside humans to perform complex, multi-step tasks efficiently and intelligently across industries like research, business automation, and customer support.




created 7 chunks
first

In [ ]:
# 3) Token Based Text splitter

print("TOKEN BASED TEXT SPLITTER")
token_splitter=TokenTextSplitter(
    chunk_size=50, # SIZE IN TOKEN AND NOT CHARACTERS
    chunk_overlap=10, # Overlap between the chunks
    )


token_chunks=token_splitter.split_text(text1)
print(text1)
print(f'created {len(recursive_chunks)} chunks')

print(f'first chunk : {token_chunks[0]}....')
print(f'second chunk : {token_chunks[1]}....')
print(f'third chunk : {token_chunks[2]}....')



TOKEN BASED TEXT SPLITTER
LLM Agents are advanced systems built on large language models (LLMs) that go beyond generating text — they can reason, plan, and take actions to achieve specific goals. Unlike traditional chatbots that only respond to prompts, LLM Agents can interact with external tools, APIs, or databases, making them capable of performing real tasks such as searching the web, analyzing data, or automating workflows.

They operate using a “think–act–learn” loop: the model interprets a goal, decides what actions to take, executes them, and updates its understanding based on the results. This enables dynamic problem-solving, not just static conversation.

In short, LLM Agents represent the next evolution of AI — from passive responders to active collaborators — capable of working autonomously or alongside humans to perform complex, multi-step tasks efficiently and intelligently across industries like research, business automation, and customer support.




created 7 chunks
fir

In [ ]:
# 📊 Comparison
print("\n📊 Text Splitting Methods Comparison:")
print("\nCharacterTextSplitter:")
print("  ✅ Simple and predictable")
print("  ✅ Good for structured text")
print("  ❌ May break mid-sentence")
print("  Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("  ✅ Respects text structure")
print("  ✅ Tries multiple separators")
print("  ✅ Best general-purpose splitter")
print("  ❌ Slightly more complex")
print("  Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("  ✅ Respects model token limits")
print("  ✅ More accurate for embeddings")
print("  ❌ Slower than character-based")
print("  Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
  ✅ Simple and predictable
  ✅ Good for structured text
  ❌ May break mid-sentence
  Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
  ✅ Respects text structure
  ✅ Tries multiple separators
  ✅ Best general-purpose splitter
  ❌ Slightly more complex
  Use when: Default choice for most texts

TokenTextSplitter:
  ✅ Respects model token limits
  ✅ More accurate for embeddings
  ❌ Slower than character-based
  Use when: Working with token-limited models
